In [1]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from tifffile import imsave, imread
import os

from binomdataset_ffhq_super_res import BinomDataset 
from gap_unet_resblock_ffhq_super_res import UN
from utils import inpainting

import pytorch_lightning as pl
from pytorch_lightning.callbacks import LearningRateMonitor, ModelCheckpoint, EarlyStopping
import torch.utils.data as dt

if not torch.cuda.is_available():
    raise ValueError("GPU not found, code will run on CPU and can be extremely slow!")
else:
    device = torch.device("cuda:0")

print(f'Device in use: {device}')

Device in use: cuda:0


In [2]:
# data = np.concatenate((imread('/kaggle/input/conv-set/trainingDataGT.tif'), imread('/kaggle/input/conv-set/testDataGT.tif')))
# # plt.imshow(data[0])
# print(data.shape)

In [3]:
def psnrToString(inp):
    if inp < 0:
        return 'm'+str(-inp)
    else:
        return str(inp)

minpsnr = -10
maxpsnr = 0

name = psnrToString(minpsnr)+"to"+psnrToString(maxpsnr)+"-512x512-ffhq-super-res"

CHECKPOINT_PATH = '/kaggle/working/checkpoints/'
os.makedirs(CHECKPOINT_PATH, exist_ok=True)
CHECKPOINT_PATH , name

('/kaggle/working/checkpoints/', 'm10to0-512x512-ffhq-super-res')

In [4]:
maxepochs = 150
# masks = inpainting(masksize = 64)
# mask = torch.from_numpy(masks.generate_mask()) /kaggle/usr/lib/utils/utils.py
dataset = BinomDataset(root = '/kaggle/input/flickrfaceshq-dataset-ffhq/', windowSize = 256, minPSNR = minpsnr, maxPSNR = maxpsnr, virtSize = 1)
# val_dataset = BinomDataset(data = data[round(data.shape[0]*0.9):], windowSize = 256, minPSNR = minpsnr, maxPSNR = maxpsnr)

In [5]:
from torch.utils.data import random_split
import random
seed = 42
torch.manual_seed(seed)
random.seed(seed)

# Set the sizes for your train and test sets
total_size = len(dataset)
train_size = int(0.8 * total_size)  # 80% for training
val_size = total_size - train_size  # Remaining 20% for testing

# Split the dataset
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [6]:
train_loader = dt.DataLoader(dataset, batch_size=12, shuffle=True, drop_last=True, pin_memory=False, num_workers=4) ## Changing the batch size from 32 to 16 to fit inside the gpu
val_loader = dt.DataLoader(val_dataset, batch_size=12, shuffle=False, drop_last=True,  pin_memory=False, num_workers=4)

trainer = pl.Trainer(default_root_dir=os.path.join(CHECKPOINT_PATH, name), gradient_clip_val=0.5,
                     accelerator="gpu",
                     max_time={'hours': 7, 'minutes': 15},
                     max_epochs=maxepochs, 
                     callbacks=[ModelCheckpoint(save_weights_only=False, mode="min", monitor="val_loss", every_n_epochs= 1),
                                LearningRateMonitor("epoch"),
                                EarlyStopping('val_loss', patience=2000)])

model = UN(channels = 3, levels=10, depth=7,start_filts=32, 
           up_mode = 'upsample', merge_mode = 'concat').to(device)

/kaggle/usr/lib/gap_unet_resblock_ffhq_super_res/gap_unet_resblock_ffhq_super_res.py:261: UserWarning: nn.init.xavier_normal is now deprecated in favor of nn.init.xavier_normal_.
  init.xavier_normal(m.weight)
/kaggle/usr/lib/gap_unet_resblock_ffhq_super_res/gap_unet_resblock_ffhq_super_res.py:262: UserWarning: nn.init.constant is now deprecated in favor of nn.init.constant_.
  init.constant(m.bias, 0)


In [7]:
trainer.fit(model, train_loader, val_loader)
trainer.save_checkpoint(os.path.join(CHECKPOINT_PATH, name)+'.ckpt')

2024-08-24 11:48:45.998143: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-08-24 11:48:45.998241: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-08-24 11:48:46.134733: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/opt/conda/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


Training: |          | 0/? [00:00<?, ?it/s]

/opt/conda/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

In [8]:
# CKPT_PATH = '/kaggle/input/m40to30-256x256-ffhq/pytorch/default/1/m40to30-256x256-ffhq.ckpt'
# model = UN.load_from_checkpoint(CKPT_PATH).to(device)

In [9]:
# trainer.fit(model,
#             train_dataloaders = train_loader,
#             val_dataloaders = val_loader,
#             ckpt_path = CKPT_PATH)
# trainer.save_checkpoint(os.path.join(CHECKPOINT_PATH, name)+'-run-2.ckpt')

In [10]:
# from kaggle.api.kaggle_api_extended import KaggleApi
# from config import username, key
# os.environ['KAGGLE_USERNAME'] = username
# os.environ['KAGGLE_KEY'] = key
# api = KaggleApi()
# api.authenticate()


In [11]:
# !pkill jupyter